<a href="https://colab.research.google.com/github/priya10241/Virality/blob/main/ExtractFeaturesVirality.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from textblob import TextBlob
from sklearn.feature_extraction.text import TfidfVectorizer

# Load original
df = pd.read_csv('Train.csv')

def get_content_only(text):
    text = str(text)
    blob = TextBlob(text)
    words = text.split()

    return pd.Series({
        'title_length': len(text),
        'title_sentiment': blob.sentiment.polarity,
        'word_count': len(words),
        'unique_word_ratio': len(set(words)) / len(words) if len(words) > 0 else 0,
        'exclamation_count': text.count('!'),
        'question_mark_present': 1 if '?' in text else 0,
        'uppercase_ratio': sum(1 for c in text if c.isupper()) / len(text) if len(text) > 0 else 0,
        'contains_numbers': 1 if any(c.isdigit() for c in text) else 0,
        'emotion_scores': blob.sentiment.subjectivity
    })

# 1. Generate Core Features
content_df = df['Post Title'].apply(get_content_only)

# 2. Generate Sentence Embeddings (20 dimensions)
tfidf = TfidfVectorizer(max_features=20, stop_words='english')
embeds = tfidf.fit_transform(df['Post Title'].fillna('')).toarray()
embed_df = pd.DataFrame(embeds, columns=[f'sentence_embedding_{i}' for i in range(20)])

# 3. Final Merge (Keep ONLY these and 'viral')
final_df = pd.concat([content_df, embed_df, df[['viral']]], axis=1)

# Save result
final_df.to_csv('final_training_data.csv', index=False)

In [ ]:
import pandas as pd
from textblob import TextBlob
from sklearn.feature_extraction.text import TfidfVectorizer

# Load original
df = pd.read_csv('test_reddit.csv')

def get_content_only(text):
    text = str(text)
    blob = TextBlob(text)
    words = text.split()

    return pd.Series({
        'title_length': len(text),
        'title_sentiment': blob.sentiment.polarity,
        'word_count': len(words),
        'unique_word_ratio': len(set(words)) / len(words) if len(words) > 0 else 0,
        'exclamation_count': text.count('!'),
        'question_mark_present': 1 if '?' in text else 0,
        'uppercase_ratio': sum(1 for c in text if c.isupper()) / len(text) if len(text) > 0 else 0,
        'contains_numbers': 1 if any(c.isdigit() for c in text) else 0,
        'emotion_scores': blob.sentiment.subjectivity
    })

# 1. Generate Core Features
content_df = df['Post Title'].apply(get_content_only)

# 2. Generate Sentence Embeddings (20 dimensions)
tfidf = TfidfVectorizer(max_features=20, stop_words='english')
embeds = tfidf.fit_transform(df['Post Title'].fillna('')).toarray()
embed_df = pd.DataFrame(embeds, columns=[f'sentence_embedding_{i}' for i in range(20)])

# 3. Final Merge (Keep ONLY these and 'viral')
final_df = pd.concat([content_df, embed_df, df[['viral']]], axis=1)

# Save result
final_df.to_csv('final_testing_data.csv', index=False)